# HW 6 — H5: obdurate (committed) splits

**My predictions (Oct 25, 2026):**
- Committed splits occur; on Full they take about 10% of round time.
- The median duration of a committed split is 5–20 s.
- The player who breaks a committed split is in the minority more often than chance (share above k/7).

Data: `load_runs("outcome", roles={"train"})`: train groups only, rounds with no bot and with an end marker (aborted 1-row rounds and unplayed waves are dropped). Topologies Full, Hierarchy, Star. Val is not used.

In [1]:
import pandas as pd
from slider import config
from slider import slider_logs as sl
from slider import h5

runs, table = sl.load_runs("outcome", roles={"train"})
pd.crosstab(table["dataset"], table["topology"], margins=True)

topology,Full,Hierarchy,Star,All
dataset,,,,
CC,31,34,33,98
CL,42,41,0,83
EL,0,33,0,33
All,73,108,33,214


## 1. Committed splits

A row is a committed split if the signs are mixed and every |s_i| ≥ 0.9. Spells use the forward hold and merge gaps shorter than τ = 1 s (`spells` in `src/slider/fraught.py`). Mean % committed is the share of each round spent in committed rows, averaged over rounds. KM median: escape = the spell ended; censored = still open when a failed round ended.

In [2]:
sp, per_run = h5.committed_spells(runs, table, cutoff=0.9, tau=1.0)
h5.topology_summary(sp, per_run).round(2)

,runs,mean % committed,spells,KM median (s),consensus %,relapse %,unresolved %,censored %
topology,,,,,,,,
Full,73,9.74,159,2.36,27.04,71.07,0.63,1.26
Hierarchy,108,11.04,340,1.54,18.24,80.29,1.18,0.29
Star,33,9.34,35,2.61,54.29,45.71,0.00,0.00


Full only, by k = minority size at the spell's last committed row.

In [3]:
full = sp[sp["topology"] == "Full"]
h5.spell_summary(full, "k", [1, 2, 3]).round(2)

,spells,KM median (s),consensus %,relapse %,unresolved %,censored %
k,,,,,,
1,74,5.00,32.43,66.22,0.00,1.35
2,53,1.32,28.30,67.92,1.89,1.89
3,32,0.94,12.50,87.50,0.00,0.00


## 2. Who breaks a committed split (Full)

Breaker = the player whose |s| drops below 0.9 in the row after the spell (spells still open at the end of a failed round are left out). If several players drop in that row the spell is ambiguous and left out. Null: the breaker is a random player, so P(minority) = k/7. 95% CI of (observed − expected) / n: 10,000 bootstrap resamples of groups (log files), seed 573.

In [4]:
pd.Series(h5.breaker_summary(full)).round(3).to_frame("Full")

,Full
spells ended,157.000
ambiguous,2.000
no drop,0.000
breaker spells (n),155.000
groups,24.000
observed minority,60.000
expected minority,38.571
observed share,0.387
expected share,0.249
(obs - exp) / n,0.138


## 3. Confound: was the breaker the last player to reach |s| ≥ 0.9 before the spell started? (null 1/7)

In [5]:
pd.Series(h5.confound_summary(full)).round(3).to_frame("Full")

,Full
breaker spells with a clear last reacher (n),153.000
breaker = last to reach,28.000
share,0.183
null share,0.143
ambiguous last reacher,2.000


## 4. Sensitivity: cutoff 0.8 / 0.9 / 1.0 × τ 0.5 / 1 / 2 s

Saved to `results/tables/h5_train.csv`. Mean % committed does not depend on τ.

In [6]:
grid = h5.sensitivity(runs, table, cutoffs=[0.8, 0.9, 1.0], taus=[0.5, 1.0, 2.0])
grid.to_csv(config.H5_TRAIN, index=False)
grid.pivot_table(index=["cutoff", "tau"], columns="topology", values=["mean % committed", "spells", "KM median (s)"]).round(2)

KM median (s)                 mean % committed                   \
topology            Full Hierarchy  Star             Full Hierarchy   Star   
cutoff tau                                                                   
0.8    0.5          1.70      1.75  1.71            12.19     13.62  11.06   
       1.0          2.26      1.80  2.28            12.19     13.62  11.06   
       2.0          3.00      3.00  2.28            12.19     13.62  11.06   
0.9    0.5          1.22      1.40  1.38             9.74     11.04   9.34   
       1.0          2.36      1.54  2.61             9.74     11.04   9.34   
       2.0          2.86      2.50  2.46             9.74     11.04   9.34   
1.0    0.5          1.20      1.00  0.85             5.47      6.96   6.79   
       1.0          1.30      1.00  1.09             5.47      6.96   6.79   
       2.0          1.81      1.28  2.31             5.47      6.96   6.79   

           spells                  
topology     Full Hierarchy  Star  
cutoff tau                         
0.8    0.5  208.0     399.0  48.0  
       1.0  185.0     367.0  42.0  
       2.0  145.0     251.0  33.0  
0.9    0.5  220.0     428.0  52.0  
       1.0  159.0     340.0  35.0  
       2.0  117.0     241.0  32.0  
1.0    0.5  178.0     358.0  38.0  
       1.0  161.0     339.0  34.0  
       2.0  121.0     255.0  24.0

In [7]:
grid.set_index(["cutoff", "tau", "topology"])[[a + " %" for a in h5.AFTER]].round(1)

consensus %  relapse %  unresolved %  censored %
cutoff tau topology                                                   
0.8    0.5 Full              24.0       73.1           1.9         1.0
           Hierarchy         16.8       82.0           1.0         0.3
           Star              41.7       58.3           0.0         0.0
       1.0 Full              27.0       69.7           2.2         1.1
           Hierarchy         18.3       80.4           1.1         0.3
           Star              47.6       52.4           0.0         0.0
       2.0 Full              34.5       61.4           2.8         1.4
           Hierarchy         26.7       71.3           1.6         0.4
           Star              60.6       39.4           0.0         0.0
0.9    0.5 Full              19.5       79.1           0.5         0.9
           Hierarchy         14.5       84.3           0.9         0.2
           Star              36.5       63.5           0.0         0.0
       1.0 Full              27.0       71.1           0.6         1.3
           Hierarchy         18.2       80.3           1.2         0.3
           Star              54.3       45.7           0.0         0.0
       2.0 Full              36.8       60.7           0.9         1.7
           Hierarchy         25.7       72.2           1.7         0.4
           Star              59.4       40.6           0.0         0.0
1.0    0.5 Full              19.7       78.7           1.1         0.6
           Hierarchy         13.4       85.2           1.4         0.0
           Star              39.5       60.5           0.0         0.0
       1.0 Full              21.7       76.4           1.2         0.6
           Hierarchy         14.2       84.4           1.5         0.0
           Star              44.1       55.9           0.0         0.0
       2.0 Full              28.9       68.6           1.7         0.8
           Hierarchy         18.8       79.2           2.0         0.0
           Star              62.5       37.5           0.0         0.0

In [8]:
grid[grid["topology"] == "Full"].set_index(["cutoff", "tau"])[
    ["breaker spells (n)", "observed minority", "expected minority", "(obs - exp) / n", "CI low", "CI high"]].round(3)

breaker spells (n)  observed minority  expected minority  \
cutoff tau                                                             
0.8    0.5               205.0               83.0             53.286   
       1.0               182.0               75.0             47.143   
       2.0               143.0               65.0             38.714   
0.9    0.5               216.0               81.0             54.143   
       1.0               155.0               60.0             38.571   
       2.0               114.0               52.0             29.714   
1.0    0.5               177.0               55.0             39.000   
       1.0               160.0               55.0             36.143   
       2.0               120.0               47.0             28.571   

            (obs - exp) / n  CI low  CI high  
cutoff tau                                    
0.8    0.5            0.145   0.076    0.218  
       1.0            0.153   0.078    0.235  
       2.0            0.184   0.105    0.267  
0.9    0.5            0.124   0.072    0.173  
       1.0            0.138   0.075    0.203  
       2.0            0.195   0.120    0.272  
1.0    0.5            0.090   0.048    0.138  
       1.0            0.118   0.068    0.174  
       2.0            0.154   0.095    0.212

## 5. Frozen analysis and checks

`h5.run(roles)` in `src/slider/h5.py` is the frozen H5 analysis (main setting + the checks below). Frozen choices:

**Data**
- Runs: `load_runs("outcome", roles=roles)`. Development here uses `roles={"train"}` only; val and test are not used.
- `outcome` drops aborted 1-row rounds, runs with fewer than 2 rows, runs with no end marker and runs with a bot. It keeps whole-second files (check 5.3 drops them).
- Topologies: Full, Hierarchy, Star. A group = one log file.

**Committed splits and spells**
- s = weights × X. A row is a committed split if the signs are mixed and every |s_i| ≥ 0.9.
- Forward hold: row k lasts [t_k, t_k+1); the last row lasts until t_end.
- t_end = last row time if the round succeeded; max(time limit, last row time) if it failed (300 s lab, 180 s cloud). Failed rounds are censored.
- Spells merge non-committed stretches shorter than τ = 1 s, unless the stretch has a consensus row. Consensus row = the last row of a successful round.
- `after` = the next event once the spell ends: consensus, relapse (new spell), unresolved (round ends first) or censored (still open when a failed round ends).
- No minimum spell duration in the main setting (check 5.2 tries 0.5 s and 1 s).

**Summaries**
- % committed = forward-hold time in committed rows / t_end, per round, averaged over rounds of a topology.
- Kaplan–Meier: escape = the spell ended (after ≠ censored); at tied times escapes count before censorings; median = first time S ≤ 0.5.
- k = minority size at the spell's last committed row. Minority / majority sides are always taken at that row.

**Breaker test (Full)**
- Breaker = the player whose |s| drops below 0.9 in the row after the spell's last committed row. Censored spells are left out. If several players drop in that row, the spell is ambiguous and left out.
- Null: the breaker is a random player, expected P(minority) = k/7. Reported: observed and expected minority counts and (observed − expected) / n.
- 95% CI: percentile bootstrap, 10,000 resamples of groups (log files), seed 573.

**Last arriver and confound checks**
- Last arriver = the player whose final stretch at |s| ≥ 0.9 started latest, up to the spell's first row. Ties: ambiguous, left out of the confound checks.
- (a) breaker is not the last arriver: expected (k − L)/6, L = 1 if the last arriver is in the minority, else 0. (b) last arriver in the majority: expected k/7.

**Other checks:** spells ≥ 0.5 s and ≥ 1 s (Full by k + breaker test); without whole-second files (main table + breaker test).

**Separate:** the sensitivity grid (part 4: cutoff 0.8 / 0.9 / 1.0 × τ 0.5 / 1 / 2 s) is `h5.sensitivity`, not part of `h5.run`.

In [9]:
res = h5.run({"train"})
assert res["main"].equals(h5.topology_summary(sp, per_run))   # same as part 1
assert res["breaker"] == h5.breaker_summary(full)              # same as part 2
checks = res["checks"]
checks.to_csv(config.H5_TRAIN_CHECKS, index=False)
print(len(checks), "rows saved to", config.H5_TRAIN_CHECKS.name)

21 rows saved to h5_train_checks.csv


### 5.1 Confound: breaker test on spells where the last arriver cannot explain a minority breaker

In [10]:
checks[checks["check"] == "confound"].dropna(axis=1, how="all").round(3)

,check,variant,part,key,breaker spells (n),groups,observed minority,expected minority,observed share,expected share,(obs - exp) / n,CI low,CI high
7,confound,"(a) breaker is not the last arriver, expected ...",breaker,Full,125.0,24.0,42.0,30.500,0.336,0.244,0.092,0.021,0.159
8,confound,"(b) last arriver in the majority, expected k/7",breaker,Full,110.0,23.0,36.0,24.429,0.327,0.222,0.105,0.038,0.174


### 5.2 Minimum spell duration: drop spells shorter than 0.5 s, then 1 s (Full)

In [11]:
checks[checks["check"] == "min duration"].dropna(axis=1, how="all").round(3)

,check,variant,part,key,spells,KM median (s),consensus %,relapse %,unresolved %,censored %,...,no drop,breaker spells (n),groups,observed minority,expected minority,observed share,expected share,(obs - exp) / n,CI low,CI high
9,min duration,spells >= 0.5 s,Full by k,1,68.0,5.451,35.294,63.235,0.000,1.471,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,min duration,spells >= 0.5 s,Full by k,2,45.0,2.029,22.222,75.556,2.222,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,min duration,spells >= 0.5 s,Full by k,3,25.0,1.238,16.000,84.000,0.000,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,min duration,spells >= 0.5 s,breaker,Full,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,135.0,24.0,55.0,32.714,0.407,0.242,0.165,0.115,0.221
13,min duration,spells >= 1.0 s,Full by k,1,63.0,6.300,31.746,66.667,0.000,1.587,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,min duration,spells >= 1.0 s,Full by k,2,34.0,2.400,17.647,79.412,2.941,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,min duration,spells >= 1.0 s,Full by k,3,15.0,2.407,6.667,93.333,0.000,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,min duration,spells >= 1.0 s,breaker,Full,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,109.0,21.0,39.0,24.571,0.358,0.225,0.132,0.079,0.183


### 5.3 Without whole-second files (coarse_time)

In [12]:
checks[checks["check"] == "no whole-second files"].dropna(axis=1, how="all").round(3)

,check,variant,part,key,runs,mean % committed,spells,KM median (s),consensus %,relapse %,...,no drop,breaker spells (n),groups,observed minority,expected minority,observed share,expected share,(obs - exp) / n,CI low,CI high
17,no whole-second files,coarse_time files dropped,topology,Full,64.0,9.939,150.0,2.244,26.667,72.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,no whole-second files,coarse_time files dropped,topology,Hierarchy,100.0,11.254,330.0,1.500,17.879,80.606,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,no whole-second files,coarse_time files dropped,topology,Star,33.0,9.340,35.0,2.605,54.286,45.714,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,no whole-second files,coarse_time files dropped,breaker,Full,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,147.0,21.0,58.0,37.286,0.395,0.254,0.141,0.073,0.209
